<a href="https://colab.research.google.com/github/zzzzzssyy/ECON3916-33674-Statistical-Machine-Learning/blob/main/Lab%2013/%5BLab_13%5D_Hedonic_Pricing_and_the_FWL_Theorem.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [4]:
import pandas as pd
import statsmodels.formula.api as smf
import matplotlib.pyplot as plt


# Step 1: Ingestion and Naive Model
url = 'https://raw.githubusercontent.com/zzzzzssyy/ECON3916-33674-Statistical-Machine-Learning/main/Data/Zillow_California_2026_Hedonic.csv'
df = pd.read_csv(url)
df

,Property_Age,Distance_to_Tech_Hub,Sale_Price
0,77.5,38.1,684100.56
1,11.0,95.1,413634.22
2,47.7,73.5,456709.35
3,61.9,60.3,624533.95
4,100.8,16.4,870137.54
...,...,...,...
995,87.7,10.1,932592.35
996,21.2,91.8,412741.12
997,96.5,14.5,880901.56
998,20.1,95.1,396659.79


In [5]:
naive_model = smf.ols("Sale_Price ~ Property_Age",data=df).fit()
print(naive_model.summary())
print("\nNaive Age Coefficient:", naive_model.params['Property_Age'])

                            OLS Regression Results                            
Dep. Variable:             Sale_Price   R-squared:                       0.757
Model:                            OLS   Adj. R-squared:                  0.757
Method:                 Least Squares   F-statistic:                     3105.
Date:                Wed, 18 Mar 2026   Prob (F-statistic):          1.26e-308
Time:                        19:53:29   Log-Likelihood:                -12818.
No. Observations:                1000   AIC:                         2.564e+04
Df Residuals:                     998   BIC:                         2.565e+04
Df Model:                           1                                         
Covariance Type:            nonrobust                                         
                   coef    std err          t      P>|t|      [0.025      0.975]
--------------------------------------------------------------------------------
Intercept     3.013e+05   7218.570     41.742   

In [6]:
# Step 2: The Multivariate Model
multi_model = smf.ols('Sale_Price ~ Property_Age + Distance_to_Tech_Hub', data=df).fit()
print(multi_model.summary())
print("\nMultivariate Age Coefficient:", multi_model.params['Property_Age'])

                            OLS Regression Results                            
Dep. Variable:             Sale_Price   R-squared:                       0.954
Model:                            OLS   Adj. R-squared:                  0.954
Method:                 Least Squares   F-statistic:                 1.040e+04
Date:                Wed, 18 Mar 2026   Prob (F-statistic):               0.00
Time:                        19:57:03   Log-Likelihood:                -11982.
No. Observations:                1000   AIC:                         2.397e+04
Df Residuals:                     997   BIC:                         2.399e+04
Df Model:                           2                                         
Covariance Type:            nonrobust                                         
                           coef    std err          t      P>|t|      [0.025      0.975]
----------------------------------------------------------------------------------------
Intercept             1.203e+06 

In [7]:
# Step 3: FWL Theorem Manual Proof
# 3a: Partial out distance from Price
res_y_model = smf.ols('Sale_Price ~ Distance_to_Tech_Hub', data=df).fit()
df['Price_Residuals'] = res_y_model.resid
df['Price_Residuals']

,Price_Residuals
0,-56823.332740
1,19000.990249
2,-69149.815200
3,18481.157582
4,-2619.815668
...,...
995,21560.763160
996,-1940.516556
997,-3398.817768
998,2026.560249


In [9]:
# 3b: Partial out distance from Age
res_x_model = smf.ols('Property_Age ~ Distance_to_Tech_Hub', data=df).fit()
df['Age_Residuals'] =res_x_model.resid
df['Age_Residuals']

,Age_Residuals
0,0.621803
1,-13.689856
2,3.233510
3,5.347789
4,4.053610
...,...
995,-14.814575
996,-6.511286
997,-1.986001
998,-4.589856


In [10]:
df

,Property_Age,Distance_to_Tech_Hub,Sale_Price,Price_Residuals,Age_Residuals
0,77.5,38.1,684100.56,-56823.332740,0.621803
1,11.0,95.1,413634.22,19000.990249,-13.689856
2,47.7,73.5,456709.35,-69149.815200,3.233510
3,61.9,60.3,624533.95,18481.157582,5.347789
4,100.8,16.4,870137.54,-2619.815668,4.053610
...,...,...,...,...,...
995,87.7,10.1,932592.35,21560.763160,-14.814575
996,21.2,91.8,412741.12,-1940.516556,-6.511286
997,96.5,14.5,880901.56,-3398.817768,-1.986001
998,20.1,95.1,396659.79,2026.560249,-4.589856


In [12]:
# 3c: Regress Residuals on Residuals (-1 removes the intercept for exact mathematical matching)
fwl_model = smf.ols('Price_Residuals ~ Age_Residuals', data=df).fit()
print("\nFWL Isolated Age Coefficient:", fwl_model.params['Age_Residuals'])


FWL Isolated Age Coefficient: -2063.129216802138


In [14]:
import numpy as np
import pandas as pd
import statsmodels.formula.api as smf
import plotly.graph_objects as go

# ── Load data (same source as Lab 13) ─────────────────────────────────────────
url = 'https://raw.githubusercontent.com/zzzzzssyy/ECON3916-33674-Statistical-Machine-Learning/main/Data/Zillow_California_2026_Hedonic.csv'
df = pd.read_csv(url)

# ── Fit the multivariate OLS model ────────────────────────────────────────────
multi_model = smf.ols('Sale_Price ~ Property_Age + Distance_to_Tech_Hub', data=df).fit()

# Extract coefficients from statsmodels' .params Series (indexed by variable name)
intercept = multi_model.params['Intercept']             # β₀
b_age     = multi_model.params['Property_Age']          # β₁
b_dist    = multi_model.params['Distance_to_Tech_Hub']  # β₂

print(f"Intercept (β₀)          : ${intercept:,.2f}")
print(f"β₁ (Property_Age)       : ${b_age:,.4f} per year")
print(f"β₂ (Distance_to_Hub)    : ${b_dist:,.4f} per mile")
print(f"R²                      :  {multi_model.rsquared:.4f}")

# ── Build the meshgrid for the regression surface ─────────────────────────────
# np.linspace creates 60 evenly-spaced points across each variable's observed range.
# np.meshgrid converts those two 1-D arrays into 2-D grids so that every
# (age, distance) combination is represented — this forms the "net" of the plane.
age_range  = np.linspace(df['Property_Age'].min(),         df['Property_Age'].max(),         60)
dist_range = np.linspace(df['Distance_to_Tech_Hub'].min(), df['Distance_to_Tech_Hub'].max(), 60)

age_grid, dist_grid = np.meshgrid(age_range, dist_range)

# Evaluate the OLS equation at every grid point: ŷ = β₀ + β₁·Age + β₂·Dist
price_grid = intercept + b_age * age_grid + b_dist * dist_grid

# ── Compute residuals (vertical distance from each observed point to the plane) ─
df['Residual'] = multi_model.resid   # positive = above plane, negative = below

# ── Build the Plotly 3D figure ────────────────────────────────────────────────
fig = go.Figure()

# Trace 1: Regression hyperplane (Surface)
fig.add_trace(go.Surface(
    x=age_grid,
    y=dist_grid,
    z=price_grid,
    colorscale='Blues',
    opacity=0.55,
    showscale=False,
    name='OLS Hyperplane',
    hovertemplate=(
        'Property Age: %{x:.1f} yrs<br>'
        'Distance to Hub: %{y:.2f} mi<br>'
        'Predicted Price: $%{z:,.0f}'
        '<extra>OLS Hyperplane</extra>'
    )
))

# Trace 2: Actual data points — coloured by residual magnitude
clip = df['Residual'].abs().quantile(0.95)   # clip extremes to keep colour scale readable
fig.add_trace(go.Scatter3d(
    x=df['Property_Age'],
    y=df['Distance_to_Tech_Hub'],
    z=df['Sale_Price'],
    mode='markers',
    marker=dict(
        size=4,
        color=df['Residual'],
        colorscale='RdYlGn',
        cmin=-clip, cmax=clip,
        colorbar=dict(
            title=dict(text='Residual ($)', side='right'),
            thickness=14, len=0.6
        ),
        opacity=0.85,
        line=dict(width=0)
    ),
    name='Observed Sales',
    hovertemplate=(
        'Property Age: %{x:.1f} yrs<br>'
        'Distance to Hub: %{y:.2f} mi<br>'
        'Sale Price: $%{z:,.0f}'
        '<extra>Observed</extra>'
    )
))

# ── Layout ────────────────────────────────────────────────────────────────────
fig.update_layout(
    title=dict(
        text=(
            '<b>Hedonic Pricing — OLS Regression Hyperplane</b><br>'
            f'<sup>β₁(Age) = ${b_age:,.1f}/yr  |  '
            f'β₂(Dist) = ${b_dist:,.1f}/mi  |  '
            f'R² = {multi_model.rsquared:.3f}</sup>'
        ),
        x=0.5, xanchor='center', font_size=14
    ),
    scene=dict(
        xaxis=dict(title='Property Age (yrs)'),
        yaxis=dict(title='Distance to Tech Hub (mi)'),
        zaxis=dict(title='Sale Price ($)', tickformat=',.0f'),
        camera=dict(eye=dict(x=1.6, y=-1.6, z=0.9))
    ),
    legend=dict(x=0.02, y=0.95),
    margin=dict(l=0, r=0, t=80, b=0),
    width=900, height=650
)

fig.show()

Intercept (β₀)          : $1,202,971.27
β₁ (Property_Age)       : $-2,063.1292 per year
β₂ (Distance_to_Hub)    : $-7,964.2448 per mile
R²                      :  0.9543
